# Upload Data to Redshift - Simple Tutorial

This tutorial shows you how to upload **any pandas DataFrame** to AWS Redshift.

## What is Redshift?

Redshift is a database in the cloud where you can store large amounts of data and run SQL queries on it.

Think of it like Excel, but:
- It can handle millions of rows
- Multiple people can query it at the same time
- It's much faster for large datasets

## What You'll Learn

1. Load your pandas DataFrame
2. Connect to Redshift
3. Upload your data
4. Verify it worked

## Before You Start

Install the required libraries:

```bash
conda install -c conda-forge python-dotenv
conda install -c conda-forge pyarrow
pip install pandas psycopg2-binary
pip install 'awswrangler[redshift]'
```

**IMPORTANT:** 
- Make sure to install `awswrangler[redshift]` (with the `[redshift]` part) to get all required dependencies!
- **On Mac/zsh**: Use quotes: `pip install 'awswrangler[redshift]'`
- **On Windows/bash**: `pip install awswrangler[redshift]` (no quotes needed)

You'll also need:
- Redshift cluster host, username, and password
- An S3 bucket name (for temporary staging)

## Step 1: Import Libraries

In [1]:
import pandas as pd
import awswrangler as wr
import psycopg2
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

False

## Step 2: Load Your Data

Load your data into a pandas DataFrame. Here are common ways to load data:

In [6]:
# From a CSV file:
#df = pd.read_csv('your_file.csv')

# From an Excel file:
# df = pd.read_excel('your_file.xlsx')

# From a Parquet file:
df = pd.read_parquet('../2. Training/data_preparation_output/train.parquet')
df = df.sample(100)

# Look at your data
print(f"We have {len(df)} rows and {len(df.columns)} columns")
df.head()

We have 100 rows and 58 columns


,idconsumo,id_contaservico,codigocontaservico,idconta,iddim_date_inicio,iddim_date_fim,id_produto_actual,tipo_produto_actual,tipo_subscricao,tipo_stb,...,was_contacted,topup_count,topup_total_value,topup_avg_value,topup_std_value,topup_cv_value,topup_days_since_last,used_selfcare,topup_type_nunique,topup_channel_nunique
24667,395810153,2428346,161492160101,2372616,2024-03-28,2024-04-26,24,normal,7,HD,...,0,4,9938.60,2484.65,1283.147403,0.516430,29.0,0,1,0
10192,381866257,3270712,191478420101,3214061,2024-01-31,2024-02-25,24,normal,4,HD,...,0,13,10087.73,775.97,1155.991960,1.489738,25.0,0,3,0
6813,378558794,2523300,170159100101,2467514,2024-01-19,2024-01-21,24,normal,4,HD,...,0,18,10350.90,575.05,791.366094,1.376169,2.0,0,3,0
41624,412303463,3087469,190227560101,3030956,2024-05-30,2024-06-05,24,tafacil7,7,HD,...,0,32,11789.52,368.42,483.241988,1.311661,6.0,0,4,0
74386,483419450,875259,130982410101,849016,2025-02-07,2025-03-09,24,normal,7,HD,...,0,5,8859.64,1771.92,2426.312340,1.369313,1.0,0,2,0


## Step 3: Configure Your Redshift Details

**IMPORTANT:** Change these values to match your Redshift cluster!

**Best Practice:** Store your password in a `.env` file instead of hardcoding it:

Create a file called `.env` in your project folder:
```
REDSHIFT_PASSWORD=your-password-here
```

In [3]:
# Your Redshift cluster details - CHANGE THESE!
REDSHIFT_HOST = "redshift-cluster-dsi.cl4o4mmtx9ir.af-south-1.redshift.amazonaws.com"
REDSHIFT_PORT = 5439
REDSHIFT_DB = "dev"
REDSHIFT_USER = "awsuser"

# Password from .env file (recommended)
#REDSHIFT_PASSWORD = os.getenv('REDSHIFT_PASSWORD')

# Or hardcode it (not recommended for security)
REDSHIFT_PASSWORD = "RFMPXefkbh465%."

if not REDSHIFT_PASSWORD:
    raise RuntimeError("REDSHIFT_PASSWORD not set. Create a .env file or set the variable.")

# Table details
SCHEMA = 'public'  # Usually 'public' is fine
TABLE_NAME = 'upload_to_redshift_from_local'  # Pick any name you want

# S3 bucket for temporary files
S3_BUCKET = 's3://sagemaker-af-south-1-733246370304/upload_to_redshift_from_local/'

IAM_ROLE = "arn:aws:iam::733246370304:role/service-role/AmazonRedshift-CommandsAccessRole-20250802T101652"

print("Configuration set!")

Configuration set!


## Step 4: Connect to Redshift

This creates a connection to your Redshift database using username and password.

In [7]:
import boto3

session = boto3.Session(profile_name="AdministratorAccess-733246370304")

# Quick test
print(session.client("s3").list_buckets())

{'ResponseMetadata': {'RequestId': 'Y2YPH9KHARHGQTCD', 'HostId': '1Ernw6jDC1dIZTmDXiqs3lxMjBOdbPdmf6qNVbVGCQU0dKrEMGH2xF1ZX/7OqD7h8hZHF6VwFl4=', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': '1Ernw6jDC1dIZTmDXiqs3lxMjBOdbPdmf6qNVbVGCQU0dKrEMGH2xF1ZX/7OqD7h8hZHF6VwFl4=', 'x-amz-request-id': 'Y2YPH9KHARHGQTCD', 'date': 'Thu, 13 Nov 2025 16:55:53 GMT', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'Buckets': [{'Name': 'aws-cloudtrail-logs-733246370304-c1842f05', 'CreationDate': datetime.datetime(2025, 8, 25, 10, 34, 18, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::aws-cloudtrail-logs-733246370304-c1842f05'}, {'Name': 'aws-glue-assets-733246370304-af-south-1', 'CreationDate': datetime.datetime(2025, 5, 9, 11, 6, tzinfo=tzutc()), 'BucketArn': 'arn:aws:s3:::aws-glue-assets-733246370304-af-south-1'}, {'Name': 'aws-glue-compaction-manifest-733246370304-af-south-1', 'CreationDate': datetime.datetime(2025, 7, 14, 8, 7, 24, tz

In [4]:
# Connect to Redshift
print("Connecting to Redshift...")

conn = psycopg2.connect(
    host=REDSHIFT_HOST,
    port=REDSHIFT_PORT,
    dbname=REDSHIFT_DB,
    user=REDSHIFT_USER,
    password=REDSHIFT_PASSWORD
)

print("Connected successfully!")

Connecting to Redshift...
Connected successfully!


## Step 5: Upload Your Data

This is the main step - it uploads your DataFrame to Redshift!

What happens:
1. Your data is saved to S3 temporarily
2. A table is created in Redshift (if it doesn't exist)
3. Data is copied from S3 to Redshift (very fast!)

In [10]:
# Upload the data
print(f"Uploading {len(df)} rows to Redshift...")

wr.redshift.copy(
    df=df,                    # Your DataFrame
    path=S3_BUCKET,          # Temporary S3 location
    con=conn,                # The connection we created
    schema=SCHEMA,           # Schema name
    table=TABLE_NAME,        # Table name
    mode='append',           # 'append' = add rows, 'overwrite' = replace all data
    boto3_session=session
)

print("Upload complete!")

Uploading 100 rows to Redshift...
Upload complete!


**What does `mode` mean?**

- `mode='append'` - Adds your data to the table (keeps existing data)
- `mode='overwrite'` - Deletes everything in the table and adds your data

## Step 7: Close the Connection

Always close the connection when you're done!

In [13]:
conn.close()
print("Connection closed")

Connection closed
